In [ ]:
import subprocess
import sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    ROOT = Path("/content/marl-tsc")
    if not ROOT.exists():
        subprocess.run([
            "git", "clone", "--branch", "main-flow-breakup",
            "--single-branch", "https://github.com/abergh18/marl-tsc.git", str(ROOT),
        ], check=True)
    sys.path.insert(0, str(ROOT / "src"))
else:
    ROOT = next(
        folder for folder in (Path.cwd(), *Path.cwd().parents)
        if (folder / "src" / "marl_tsc" / "notebook_setup.py").exists()
    )
    sys.path.insert(0, str(ROOT / "src"))

from marl_tsc.notebook_setup import setup_notebook

setup_notebook(IN_COLAB, ROOT)

In [ ]:
import matplotlib.pyplot as plt

from marl_tsc.baselines import fixed_time_actions, random_actions
from marl_tsc.evaluate_gifting import print_gifting_summary
from marl_tsc.mappo import train_mappo
from marl_tsc.network_types import GridNetwork
from marl_tsc.simulation_generator import SimulationGenerator
from marl_tsc.training import (
    evaluate_policies,
    evaluation_results_table,
    plot_training_histories,
)

In [ ]:
EPISODE_STEPS = 600
SECONDS_PER_ACTION = 5
SIMULATION_DURATION = EPISODE_STEPS * SECONDS_PER_ACTION
TRAFFIC_SPAWN_DURATION = int(SIMULATION_DURATION * 1.20)
TOTAL_TIMESTEPS = 300_000
EVALUATION_EPISODES = 3

# The same environment settings are used for training and evaluation.
ENV_KWARGS = {
    "green_phase_count": None,
    "min_green_seconds": 10,
    "seconds_per_action": SECONDS_PER_ACTION,
    "switch_penalty": 0.1,
    "collect_global_metrics": True,
    "global_metric_interval": 10,
}

OUTPUT_DIR = ROOT / "outputs"
SIMULATION_DIR = OUTPUT_DIR / "simulation"

## 5. Seeded Experiments 4x4

In [ ]:
SAVE_TO_DRIVE = True

if IN_COLAB and SAVE_TO_DRIVE:
    from marl_tsc.notebook_setup import mount_drive_output

    OUTPUT_DIR = mount_drive_output(IN_COLAB)

In [ ]:
EXP_SWEEP = True

SEED = 42
SEED_RANGE = 9
LEARNING_RATE = 3e-3

if EXP_SWEEP:
    from marl_tsc.exp_functions import save_history, plot_variance
    import torch
    import datetime
    time_stamp = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    grid= "4x4"

    #select network for seeded evaluation
    network = GridNetwork(4)
    generator = SimulationGenerator(
      output_dir=SIMULATION_DIR,
      network=network,
      trip_begin=0,
      trip_end=TRAFFIC_SPAWN_DURATION,
      trip_period=3,
      seed=SEED,
    )

    paths = generator.generate_all()
    traffic_light_ids = list(paths.traffic_light_ids)

    histories_mappo = []
    histories_rs   = []
    evaluation_histories = []

    #Run seeded experiment sweeps
    for seed in range(SEED, SEED + SEED_RANGE):
          torch.manual_seed(seed)
          print(f"|Seed: {seed} used for sweep|")
          MAPPO_KWARGS = {
              "config_file": paths.config_file,
              "traffic_light_ids": traffic_light_ids,
              "output_dir": OUTPUT_DIR,
              "total_timesteps": TOTAL_TIMESTEPS,
              "rollout_steps": 256,
              "max_steps": EPISODE_STEPS,
              "seed": seed,
              "learning_rate": LEARNING_RATE,
              "env_kwargs": ENV_KWARGS,
          }

          mappo_model, mappo_history, mappo_model_path = train_mappo(
              **MAPPO_KWARGS,
              use_peer_reward=False,
          )
          reward_sharing_model, reward_sharing_history, reward_sharing_model_path = train_mappo(
              **MAPPO_KWARGS,
              use_peer_reward=True,
          )
          #Append and save
          histories_mappo.append(mappo_history)#accumulate in memory
          histories_rs.append(reward_sharing_history)
          save_history(mappo_history, seed, f"mappo_{grid}_@{TOTAL_TIMESTEPS}steps_lr={LEARNING_RATE}_{time_stamp}", OUTPUT_DIR)#save to as backup
          save_history(reward_sharing_history, seed, f"rs__{grid}_@{TOTAL_TIMESTEPS}steps_lr={LEARNING_RATE}_{time_stamp}", OUTPUT_DIR)

          #Display
          fig, ax = plot_training_histories({
          "MAPPO": mappo_history,
          "Reward-sharing MAPPO": reward_sharing_history,
          })
          plt.show()

          #Evaluate
          policies = {
          "Random": random_actions,
          "Fixed-Time": fixed_time_actions,
          "MAPPO": mappo_model,
          "Reward-sharing MAPPO": reward_sharing_model,
          }

          policy_results = evaluate_policies(
              config_file=paths.config_file,
              traffic_light_ids=traffic_light_ids,
              policies=policies,
              episodes=EVALUATION_EPISODES,
              max_steps=EPISODE_STEPS,
              seed=seed,
              env_kwargs=ENV_KWARGS,
          )
          evaluation_histories.append(policy_results)
          save_history(policy_results, seed, f"evaluation_{grid}_@{TOTAL_TIMESTEPS}steps_lr={LEARNING_RATE}_{time_stamp}", OUTPUT_DIR)

          display(evaluation_results_table(policy_results))

    fig, ax = plt.subplots(figsize=(12, 5))
    plot_variance(histories_mappo, "MAPPO",              "#1565C0", ax)
    plot_variance(histories_rs,   "Reward-sharing MAPPO", "#C62828", ax)
    ax.set_xlabel("Timestep")
    ax.set_ylabel("Mean training reward")
    ax.legend()
    ax.set_title(f"Training variance across {SEED_RANGE} seeds")
    plt.tight_layout()
    plt.show()

### Results (Loaded from Drive)

In [ ]:
LOAD_FROM_DRIVE = False
if LOAD_FROM_DRIVE:
  from google.colab import drive
  drive.mount('/content/drive')
  OUTPUT_DIR = '/content/drive/MyDrive/Uni-Masters/Group Project/outputs/exp_histories'

  import json
  from pathlib import Path

  # Your OUTPUT_DIR should already point to Drive
  output_dir = Path(OUTPUT_DIR)

  # Load all histories by type
  def load_histories(pattern):
      paths = sorted(output_dir.glob(pattern))
      histories = []
      for p in paths:
          with open(p) as f:
              histories.append(json.load(f))
      print(f"Loaded {len(histories)} histories matching '{pattern}'")
      return histories

  # Training histories
  histories_mappo = load_histories("history_mappo_4x4_@300000steps*.json")
  histories_rs    = load_histories("history_rs__4x4_@300000steps*.json")
  evaluation_histories  = load_histories("history_evaluation_4x4_@300000steps*.json")

  print(f"MAPPO seeds: {len(histories_mappo)}")
  print(f"RS seeds:    {len(histories_rs)}")
  print(f"Eval seeds:  {len(evaluation_histories)}")

#### Training Graphs (Seed by Seed)

In [ ]:
if LOAD_FROM_DRIVE:
    #4x4
    for seed_idx, (mappo_hist, rs_hist) in enumerate(zip(histories_mappo, histories_rs)):
      seed = 42 + seed_idx
      print(f"\nSeed {seed}")

      # Training plots
      fig, ax = plot_training_histories({
          "MAPPO": mappo_hist,
          "Reward-sharing MAPPO": rs_hist,
      })

      plt.show()

#### Evaluation Runs

In [ ]:
#eval load

### Cross Seed Analysis

#### Variance Graph: Training

In [ ]:
from marl_tsc.exp_functions import plot_variance
smooth = 200
metric = "mean_training_reward"
fig, ax = plt.subplots(figsize=(12, 5))
plot_variance(histories_mappo, "MAPPO",              "#1565C0", ax, metric=metric,smooth=smooth)
plot_variance(histories_rs,   "Reward-sharing MAPPO", "#C62828", ax, metric=metric, smooth=smooth)
ax.set_xlabel("Timestep")
ax.set_ylabel(f"{metric}")
ax.legend()
ax.set_title(f"Training variance across {len(histories_rs)} seeds")
plt.tight_layout()
plt.show()

#### Statistical Significance: Training

In [ ]:
EXP_ANALYSIS = True
if EXP_ANALYSIS:
    from scipy import stats
    import numpy as np

    def training_auc(history, metric="mean_training_reward"):
        values = [h[metric] for h in history if metric in h]
        return float(np.trapezoid(values))


    def compare_training_histories(
        histories_a,
        histories_b,
        label_a="MAPPO",
        label_b="Reward-sharing MAPPO",
        metric="mean_training_reward",
        test_type="wilk",
    ):
        print(f"\nMetric: {metric}")
        print("=" * 60)

        # AUC
        aucs_a = [training_auc(h, metric) for h in histories_a]
        aucs_b = [training_auc(h, metric) for h in histories_b]
        print(f"\nAUC (total area under training curve):")
        print(f"  {label_a}: {np.mean(aucs_a):.2f} ± {np.std(aucs_a):.2f}")
        print(f"  {label_b}: {np.mean(aucs_b):.2f} ± {np.std(aucs_b):.2f}")
        if test_type == "wilk":
            stat, p = stats.wilcoxon(aucs_a, aucs_b)
        else:
            stat, p = stats.ttest_rel(aucs_a, aucs_b)

        print(f"  W={stat:.3f}, p={p:.4f}")

        if p < 0.05:
            print(f"  {label_a} and {label_b} show significantly different results")
        else:
            print(f"  {label_a} and {label_b} do not show significantly different results")

    compare_training_histories(
        histories_a=histories_mappo,
        histories_b=histories_rs,
        metric="mean_training_reward",
    )

#### Overall Performance Metrics: Evaluation

In [ ]:
if EXP_ANALYSIS:
  import numpy as np
  import pandas as pd
  metrics = ["mean_total_reward", "mean_local_queue", "mean_waiting_time",
            "mean_max_waiting_time", "mean_total_time_loss"]

  rows = []
  for policy_name in ["Random", "Fixed-Time", "MAPPO", "Reward-sharing MAPPO"]:
      row = {"Policy": policy_name}
      for metric in metrics:
          values = [e[policy_name][metric] for e in evaluation_histories]
          row[f"{metric}_mean"] = np.mean(values)
          row[f"{metric}_std"]  = np.std(values)
      rows.append(row)

  df = pd.DataFrame(rows)
  print('Overall Performance Accross 9 seeds')
  display(df)

#### Statistical Significance Tests: Evaluation

In [ ]:
if EXP_ANALYSIS:
  # Extract metric across seeds for each policy
  def perform_statisical_tests(metric, evaluation_histories, test_type):
    from scipy import stats
    mappo_grouping = [e["MAPPO"][metric] for e in evaluation_histories]
    rs_grouping   = [e["Reward-sharing MAPPO"][metric] for e in evaluation_histories]

    # Paired t-test — paired because same seed = same traffic conditions
    if test_type == "paired_t":
      _stat, p_value = stats.ttest_rel(mappo_grouping, rs_grouping)
      #print(f"t={_stat:.5f}, p={p_value:.4f}")
    elif test_type == 'wilk':
      _stat, p_value = stats.wilcoxon(mappo_grouping, rs_grouping)
      #print(f"W={_stat:.5f}, p={p_value:.4f}")
    else:
      raise ValueError(f"Unknown test type: {test_type}")

    return _stat, p_value

  metrics = ["mean_total_reward", "mean_local_queue", "mean_waiting_time",
            "mean_max_waiting_time", "mean_total_time_loss"]
  for metric in metrics:
    print(f"Metric) {metric}:")

    w, p = perform_statisical_tests(metric, evaluation_histories, "wilk")
    print(f"    Wilcoxon: t={w:.5f}, p={p:.4f}")
    if p < 0.05:
      print(f"    {metric} shows significantly different results between groups")
    else:
      print(f"    {metric} does not show significantly different results between groups ")
    print('')


In [ ]:
#print( f"Keys for mappo history: {histories_mappo[0][0].keys()}")

In [ ]:
#print(f"Keys for rs-mappo history: {histories_rs[0][0].keys()}")

In [ ]:
#print(f'''Evaluation history Keys:
#{evaluation_histories[0]['MAPPO'].keys()}''')

In [ ]:
!git status
#!git add .
commit = False
if commit:
  import shlex
  commit_msg = '''
  Updated to reflect gifts recieved as well as given
  '''
  !git config --global user.email user_email
  !git config --global user.name 'IsaacFayle-Waters'
  !git commit -m {shlex.quote(commit_msg)}

In [ ]:
#!git pull --rebase origin zak-pre-experiment

In [ ]:
#!git remote set-url origin https://{token}@github.com/abergh18/marl-tsc.git
#!git push -u origin zak-pre-experiment